In [1]:
from utilities import save_dataset, retrieve_datasets

save_dataset()
dataset, train_dataset, val_dataset, test_dataset = retrieve_datasets()
train_subset = train_dataset.select(range(50_000))
train_subset.save_to_disk('../data/train_rag')


/Users/dipeshpaneru/Documents/UniveristyWork/Advanced AI/Course-Assistant/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Saving the dataset (1/1 shards): 100%|██████████| 50000/50000 [00:00<00:00, 488028.69 examples/s]


In [2]:
from models import get_fine_tuned_model
fine_tuned_model, tokenizer = get_fine_tuned_model()

/Users/dipeshpaneru/Documents/UniveristyWork/Advanced AI/Course-Assistant/venv/lib/python3.10/site-packages/peft/config.py:162: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'corda_config', 'ensure_weight_tying', 'peft_version', 'qalora_group_size', 'target_parameters', 'trainable_token_indices', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


In [3]:
# from rag import RAG
# rag = RAG()
# rag.add_dataset('../data/train_rag', source_tag='train')

In [4]:
from rag_inference import RAGInference
from rag import RAG


#Getting a subset from the test 
test_subset  = test_dataset.shuffle(seed=42).select(range(200))

collected = RAGInference(model=fine_tuned_model, tokenizer=tokenizer, rag=RAG(), top_k=3).collect_outputs(test_subset, stage_name="rag")

✅ RAG pipeline ready — 50000 docs in store


In [6]:
from evaluation import Evaluation
evaluator = Evaluation()
results = evaluator.evaluate_results(fine_tuned_model, tokenizer, collected) # Get the result of various matrices for each oputput

In [7]:
results = evaluator.human_eval(results)


HUMAN EVALUATION - Sample of 15 examples
QUESTION: What is lymphosarcoma in fish, and what diagnostic methods are used to confirm it?
MODEL OUTPUT:
**Question**: How does lymphosarcma affect the eyes in fishes like the golden molly (*Poecilia reticulata*)? And why would you suspect something was amiss if your fish had an enlargement that wouldn't budge after two days when you tried to open his mouth without success under high pressure? Would you consider going for a CT scan before making any conclusion about whether he might have cancer? Is there anything else I could do besides wait until next weekend while healing naturally but getting him out of pain?

---

### **Step-by-step explanation below (based on specific questions):**

#### Step #1: Understand Lymphosarco­mas in Fishes
In fishes, lymphomas are rare neoplasms caused by viruses or bacteria. They often appear on soft tissues or organs where they release cytokines, hormones
Score each dimension 1-5:
  Relevance  — Did it addres

In [8]:
summary = evaluator.compute_averages(results)


📊 Baseline Evaluation Summary
  Perplexity:      6.51
  ROUGE-1:         0.1632
  ROUGE-2:         0.0202
  ROUGE-L:         0.0974
  Repetition Rate: 0.0113
  Repetition Rate higher than 1.5: 0
  Question Count:  0.39
  Human Evaluation:  2.57


In [10]:
from utilities import save_logs
save_logs("RAG", results, summary)


✅ Logs saved to outputs/


In [ ]:
# Could be used later

# from rag import RAGPipeline
# from rag_inference import RAGInference

# rag       = RAGPipeline()
# rag_model = RAGInference(ft_model, ft_tokenizer, rag, top_k=3)

# # Single question
# answer = rag_model.generate("What is gradient descent?")
# print(answer)

# # Full evaluation
# rag_collected = rag_model.collect_outputs(test_subset, stage_name="rag")

In [5]:
# Run this in a notebook cell to check what's in ChromaDB
import sys
sys.path.insert(0, '../src')
from rag import RAG

rag = RAG()

# Check what sources exist
results = rag.collection.get(where={"source": "pdf"})
print("PDF docs:", len(results['ids']))
print("Sample:", results['documents'])

✅ RAG pipeline ready — 50004 docs in store
PDF docs: 4
Sample: ['Dipesh paneru is studying Computer Science. He has 5 months left until graduation.', 'Rajesh Hamal is one of the most iconic and influential figures in the history of Nepali cinema, often referred to as the “Maha Nayak,” or “Great Hero.” Rising to fame in the late 1980s and 1990s, he became synonymous with the golden era of the Nepali film industry, starring in numerous blockbuster films that showcased his charismatic screen presence, distinctive voice, and versatile acting skills. Known for his professionalism and dedication, Hamal has played a wide range of roles, from action heroes to complex dramatic characters, earning him widespread admiration and respect. Beyond acting, he is also recognized for his', 'intellectual personality, often engaging in social and cultural discussions, which has further cemented his status as not just a film star but a respected public figure in Nepal.', "governance. Wikipedia As mayor, he

In [6]:
query = "who is balendra shah"
results = rag.collection.query(
    query_embeddings=rag.embedder.encode([query]).tolist(),
    n_results=3,
    where={"source": "pdf"}
)
print(results['documents'][0])

["governance. Wikipedia As mayor, he cut through bureaucratic red tape to improve waste management, education, and healthcare. Time Following the 2025 Gen Z protests that toppled Nepal's government amid widespread anger over corruption and poor governance, Shah and his Rastriya Swatantra Party swept the March 2026 general elections. Time He was sworn in as Nepal's Prime Minister on March 27, 2026, making him, at 35 years old, the youngest serving head of government in the world. Wikipedia", 'Rajesh Hamal is one of the most iconic and influential figures in the history of Nepali cinema, often referred to as the “Maha Nayak,” or “Great Hero.” Rising to fame in the late 1980s and 1990s, he became synonymous with the golden era of the Nepali film industry, starring in numerous blockbuster films that showcased his charismatic screen presence, distinctive voice, and versatile acting skills. Known for his professionalism and dedication, Hamal has played a wide range of roles, from action hero